# Phase VI Part 1: biological function and pathway enrichment analysis 
+ In this notebook, DICE genes that were identified via betweennnes which used distance as weight and eigenvector centrality which used correlation as weight will be analyzed for enrichment analysis. 

In [68]:
import gseapy as gp
import pandas as pd
import json

In [70]:
dice_genes = pd.read_csv("../data/results/dice_genes.tsv", sep="\t", usecols=[0])
dice_genes = list(dice_genes["Unnamed: 0"])
n = len(dice_genes)

In [38]:
# Upload GO gmt file
go_set = "../data/enrichment/go.gmt"
kegg_set = "../data/enrichment/kegg.gmt"

In [113]:
# Upload the GO Terms and KEGG Term Mappings
kegg_map = open("../data/enrichment/kegg_mapping.json")
go_map = open(("../data/enrichment/go_mapping.json"))
kegg_terms_mapping = json.load(kegg_map)
go_term_mapping = json.load(go_map)

In [40]:
enr_go = gp.enrich(gene_list=dice_genes, # or gene_list=glist
                 gene_sets=go_set, 
                 background=None,
                 outdir=None,
                 verbose=True)

2026-03-05 11:19:36,707 [INFO] User defined gene sets is given: ../data/enrichment/go.gmt
2026-03-05 11:19:36,856 [INFO] Run: go.gmt
2026-03-05 11:19:46,141 [INFO]   Background is not set! Use all 19591 genes in go.gmt.
2026-03-05 11:19:47,824 [INFO] Done.


In [144]:
enr_go.results.head()

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
0,go.gmt,GO:0000014,1/12,0.563413,0.712812,1.824340,1.046701,DCLRE1C
1,go.gmt,GO:0000018,21/141,0.000435,0.004136,2.519214,19.500262,MLH1;PTPRC;TP53BP1;MBTD1;SKP2;IL7R;RAD51;SMARC...
2,go.gmt,GO:0000019,1/7,0.383307,0.597461,3.228561,3.095928,MLH1
3,go.gmt,GO:0000022,1/12,0.563413,0.712812,1.824340,1.046701,RACGAP1
4,go.gmt,GO:0000027,8/29,0.000466,0.004371,5.556332,42.629571,MDN1;MRM2;NOP2;BOP1;RPL5;RPF2;RPL11;RPL24


In [145]:
# Save the results to a new df
results_go = enr_go.results
results_go.head()

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
0,go.gmt,GO:0000014,1/12,0.563413,0.712812,1.824340,1.046701,DCLRE1C
1,go.gmt,GO:0000018,21/141,0.000435,0.004136,2.519214,19.500262,MLH1;PTPRC;TP53BP1;MBTD1;SKP2;IL7R;RAD51;SMARC...
2,go.gmt,GO:0000019,1/7,0.383307,0.597461,3.228561,3.095928,MLH1
3,go.gmt,GO:0000022,1/12,0.563413,0.712812,1.824340,1.046701,RACGAP1
4,go.gmt,GO:0000027,8/29,0.000466,0.004371,5.556332,42.629571,MDN1;MRM2;NOP2;BOP1;RPL5;RPF2;RPL11;RPL24


In [146]:
# Filter out results with less than 0.05 adjusted p value
results_go_filt = results_go[results_go['Adjusted P-value'] < 0.05].copy()
print(f"Number of significant results: {len(results_go_filt)}")
results_go_filt.head()

Number of significant results: 1490


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
1,go.gmt,GO:0000018,21/141,0.000435,0.004136,2.519214,19.500262,MLH1;PTPRC;TP53BP1;MBTD1;SKP2;IL7R;RAD51;SMARC...
4,go.gmt,GO:0000027,8/29,0.000466,0.004371,5.556332,42.629571,MDN1;MRM2;NOP2;BOP1;RPL5;RPF2;RPL11;RPL24
5,go.gmt,GO:0000028,8/24,0.000108,0.001275,7.242051,66.170493,PRKDC;RPS27;FAU;RPS27L;RPS6;RPSA;RPS14;METTL17
9,go.gmt,GO:0000049,18/81,0.000005,0.000086,4.116810,50.292394,NSUN2;EEF1A1;PUS1;EIF2A;TRMT1;TARBP1;EIF2AK4;F...
11,go.gmt,GO:0000055,3/7,0.008459,0.045768,10.899323,52.017837,RAN;SDAD1;NUP88


In [147]:
# Map GO terms id to descriptions
results_go_filt["GO Term Name"] = results_go_filt["Term"].map(go_term_mapping)

In [148]:
# Add new overlap column and calculate the overlap ratio
results_go_filt["new_overlap"] = results_go_filt["Overlap"].str.split("/").str[0] + "/1132"
results_go_filt["new_overlap_rat"] = results_go_filt["Overlap"].apply(lambda x: int(x.split("/")[0]) / n)

# Reorder the df
results_go_filt = results_go_filt.iloc[:, [0, 1, 8, 9, 10,2,3,4,5,6,7]]

In [149]:
# Sort the results based on new overlap ratio
results_go_filt.sort_values("new_overlap_rat", ascending=False, inplace=True)

In [150]:
# Check the final df 
results_go_filt.head()

,Gene_set,Term,GO Term Name,new_overlap,new_overlap_rat,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
7636,go.gmt,GO:1990904,ribonucleoprotein complex,263/1132,0.197447,263/1144,7.704071e-77,2.008194e-73,4.980655,872.896156,DDX39B;MRPS22;SNRPF;LSM5;HNRNPA1;ZNHIT3;MRPS18...
883,go.gmt,GO:0005739,mitochondrion,262/1132,0.196697,262/1782,9.631977e-37,4.430710e-34,2.768276,229.574712,MRPS22;MSRA;RPUSD3;NAIF1;MTIF3;NR3C1;BCS1L;UQC...
4224,go.gmt,GO:0043933,protein-containing complex organization,259/1132,0.194444,259/1964,1.947192e-28,5.075681e-26,2.405961,153.514751,DDX39B;CHMP5;SNRPF;AIMP2;BCS1L;LSM5;MTPN;NCK1;...
6842,go.gmt,GO:0140513,nuclear protein-containing complex,257/1132,0.192943,257/1445,2.681752e-51,2.097130e-48,3.526056,410.592890,DDX39B;CCNC;PCF11;SNRPF;TADA3;LSM5;HNRNPA1;BCC...
1071,go.gmt,GO:0006396,RNA processing,234/1132,0.175676,234/1459,1.175032e-38,6.125834e-36,3.040841,265.577784,DDX39B;SGF29;PCF11;RPUSD3;TADA3;LSM5;SNRPF;HNR...


In [ ]:
# # Save GO encrichment analysis results
# results_go_filt.to_csv("../data/enrichment/dice_v1_go_results.tsv", sep="\t", index=False)

### Part 2: KEGG pathway enrichment analysis

In [151]:
enr_kegg = gp.enrich(gene_list=dice_genes, 
                 gene_sets=kegg_set, 
                 background=None,
                 outdir=None,
                 verbose=True)

2026-03-05 13:37:10,470 [INFO] User defined gene sets is given: ../data/enrichment/kegg.gmt
2026-03-05 13:37:10,520 [INFO] Run: kegg.gmt
2026-03-05 13:37:10,553 [INFO]   Background is not set! Use all 8011 genes in kegg.gmt.
2026-03-05 13:37:10,615 [INFO] Done.


In [152]:
# Save the results to a new df
results_kegg = enr_kegg.results
results_kegg.head()

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
0,kegg.gmt,hsa00010,8/68,0.385450,0.666302,1.245102,1.187009,AKR1A1;GAPDH;ACSS2;ENO3;LDHA;HK1;ALDOA;ENO2
1,kegg.gmt,hsa00020,1/30,0.959906,1.000000,0.448678,0.018360,FH
2,kegg.gmt,hsa00030,6/30,0.077106,0.200218,2.357200,6.040485,TALDO1;TKT;G6PD;PGD;PRPS1;ALDOA
3,kegg.gmt,hsa00040,1/34,0.973914,1.000000,0.394884,0.010438,AKR1A1
4,kegg.gmt,hsa00051,4/33,0.434607,0.682569,1.351022,1.125824,ALDOA;ENOSF1;TSTA3;HK1


In [153]:
# Filter out results with less than 0.05 adjusted p value
results_kegg_filt = results_kegg[results_kegg['Adjusted P-value'] < 0.05].copy()
print(f"Number of significant results: {len(results_kegg_filt)}")
results_kegg_filt.head()

Number of significant results: 87


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
14,kegg.gmt,hsa00190,31/133,6.766341e-06,1.053596e-04,2.787061,33.175926,COX7A2L;UQCRQ;ATP6V0D1;UQCR10;NDUFA6;COX7A2;ND...
71,kegg.gmt,hsa01521,16/79,5.158654e-03,2.226141e-02,2.324898,12.245424,RPS6;MAPK3;RPS6KB1;PRKCB;PLCG1;EIF4E2;GSK3B;GR...
75,kegg.gmt,hsa03008,32/108,1.251214e-08,5.523215e-07,3.871911,70.455492,RIOK2;GAR1;DKC1;RIOK1;NOL6;WDR75;WDR36;RAN;MDN...
76,kegg.gmt,hsa03010,85/158,7.938381e-44,2.452960e-41,11.377964,1129.172321,MRPS18C;MRPL24;RPS12;RPL35A;MRPL32;RPL39;RPS18...
77,kegg.gmt,hsa03013,39/180,3.142172e-06,6.472873e-05,2.543725,32.230510,DDX39B;SUMO1;EIF1;EEF1A1;NUP133;NUP160;EIF2S1;...


In [154]:
# Map GO terms id to descriptions
results_kegg_filt["KEGG Pathway"] = results_kegg_filt["Term"].map(kegg_terms_mapping)

In [155]:
# Add new overlap column and calculate the overlap ratio
results_kegg_filt["new_overlap"] = results_kegg_filt["Overlap"].str.split("/").str[0] + "/1132"
results_kegg_filt["new_overlap_rat"] = results_kegg_filt["Overlap"].apply(lambda x: int(x.split("/")[0]) / n)

# Reorder the df
results_kegg_filt = results_kegg_filt.iloc[:, [0, 1, 8, 9, 10,2,3,4,5,6,7]]

In [156]:
# Sort the results based on new overlap ratio
results_kegg_filt.sort_values("new_overlap_rat", ascending=False, inplace=True)

In [ ]:
# Check the final df
results_kegg_filt.head()

,Gene_set,Term,KEGG Pathway,new_overlap,new_overlap_rat,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
76,kegg.gmt,hsa03010,Ribosome - Homo sapiens (human),85/1132,0.063814,85/158,7.938381e-44,2.452960e-41,11.377964,1129.172321,MRPS18C;MRPL24;RPS12;RPL35A;MRPL32;RPL39;RPS18...
271,kegg.gmt,hsa05200,Pathways in cancer - Homo sapiens (human),76/1132,0.057057,76/531,1.039881e-03,5.950429e-03,1.535665,10.547942,GNAS;F2R;MLH1;CASP3;FH;CDC42;RPS6KB1;FADD;MDM2...
233,kegg.gmt,hsa05010,Alzheimer disease - Homo sapiens (human),76/1132,0.057057,76/369,7.098594e-10,7.311552e-08,2.440543,51.412363,DDIT3;CASP3;TUBB4B;UQCRQ;PPP3CB;FADD;PSMC6;EIF...
236,kegg.gmt,hsa05016,Huntington disease - Homo sapiens (human),61/1132,0.045796,61/306,1.297446e-07,4.454565e-06,2.314838,36.707999,SOD2;UQCRQ;CASP3;TUBB4B;PSMC6;UQCR10;NDUFA6;PO...
247,kegg.gmt,hsa05131,Shigellosis - Homo sapiens (human),55/1132,0.041291,55/242,4.950624e-09,2.835406e-07,2.736195,52.326317,MYL12B;VCL;CDC42;RPS6KB1;U2AF1;CTTN;MDM2;CBX3;...


In [ ]:
# # Save the kegg encrichment results
# results_kegg_filt.to_csv("../data/enrichment/dice_v1_kegg_results.tsv", sep="\t", index=False)